# 6b. Doladenie (fine-tuning) video siete R(2+1)D

V notebooku 6 bola sieť R(2+1)D zmrazená a trénoval sa iba SVM nad jej príznakmi.
Tu sieť **dotrénujeme**: poslednú vrstvu nahradíme novou s 2 výstupmi (eat / drink)
a spolu s ňou trénujeme aj posledný blok siete (`layer4`). Gradienty teda prechádzajú
z klasifikátora do konvolučných vrstiev a príznaky sa prispôsobia našej úlohe.

Prečo len posledný blok:
- prvé vrstvy rozpoznávajú všeobecné vzory (hrany, textúry, jednoduchý pohyb),
  ktoré netreba meniť,
- trénovacích klipov je málo; čím viac parametrov trénujeme, tým rýchlejšie sa sieť preučí,
- menej pamäte na GPU (vrstvy bez gradientov si neukladajú medzivýsledky).

Ďalšie nastavenia proti preučeniu: malý krok učenia, dropout pred výstupnou vrstvou,
náhodné zrkadlové otočenie klipov (augmentácia) a early stopping podľa validačnej straty.
Vrstvy BatchNorm necháme v režime inferencie, lebo pri dávke 4 klipov by ich štatistiky boli nespoľahlivé.

Hodnotenie je na rovnakých deleniach ako ostatné modely. Jedno delenie trvá na GPU
niekoľko minút; pre rýchly test možno znížiť `N_SPLITS`.

In [ ]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import GroupShuffleSplit
from torch import nn
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights

from har_utils import CLASSES, make_splits, read_frames, split_metrics, summarize, to_video_tensor

N_SPLITS = 10                        # počet delení (pre rýchly test napr. 3)
EPOCHS = 8
BATCH = 4
LR = 1e-5
TRAIN_LAYERS = ("layer4", "fc")      # trénované časti siete

df = pd.read_csv("index.csv")
y, ds = df.label.values, df.dataset.values
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"           # zmiešaná presnosť (float16) šetrí pamäť GPU

## Príprava klipov

Predspracovanie (16 snímok, 112 × 112, normalizácia Kinetics) je rovnaké ako v notebooku 3.
Klipy pripravíme raz a držíme v pamäti (float16), aby sa snímky nenačítavali v každej epoche.

In [ ]:
weights = R2Plus1D_18_Weights.KINETICS400_V1
video_transform = weights.transforms()
pretrained = r2plus1d_18(weights=weights).state_dict()     # pôvodné váhy (Kinetics-400)

clips = torch.stack([to_video_tensor(read_frames(p), video_transform)[0]
                     for p in df.path]).half()             # (klipy, 3, 16, 112, 112)
print(clips.shape)

In [ ]:
def build_model():
    model = r2plus1d_18()
    model.load_state_dict(pretrained)                       # vždy začíname z predtrénovanej siete
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(512, 2))   # nová výstupná vrstva
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith(TRAIN_LAYERS)
    return model.to(device)


def freeze_batchnorm(model):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm3d):
            m.eval()


def predict_logits(model, idx):
    model.eval()
    out = []
    with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        for b in np.array_split(idx, max(1, len(idx) // 8)):
            out.append(model(clips[b].to(device).float()).float().cpu())
    return torch.cat(out)

In [ ]:
def train_finetune(i_tr, i_va, seed=0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = build_model()

    counts = np.bincount(y[i_tr], minlength=2)                 # váhy tried
    w = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32)
    criterion = nn.CrossEntropyLoss(weight=w.to(device))
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                                 lr=LR, weight_decay=1e-4)
    scaler = torch.amp.GradScaler(enabled=use_amp)

    best_loss, best_state, best_epoch, history = np.inf, None, 0, []
    for epoch in range(EPOCHS):
        model.train()
        freeze_batchnorm(model)
        train_loss = 0
        for b in np.array_split(rng.permutation(i_tr), max(1, len(i_tr) // BATCH)):
            x = clips[b].to(device).float()
            flip = torch.rand(len(b)) < 0.5                     # augmentácia: zrkadlenie
            x[flip] = x[flip].flip(-1)
            yb = torch.tensor(y[b], device=device)

            optimizer.zero_grad()
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                loss = criterion(model(x), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * len(b)

        val_loss = criterion(predict_logits(model, i_va).to(device),
                             torch.tensor(y[i_va], device=device)).item()
        history.append((train_loss / len(i_tr), val_loss))
        if val_loss < best_loss:                                # early stopping
            best_loss, best_state, best_epoch = val_loss, copy.deepcopy(model.state_dict()), epoch + 1

    model.load_state_dict(best_state)
    return model, np.array(history), best_epoch

In [ ]:
rows, y_all, p_all, histories = [], [], [], []

for seed, tr, te in make_splits(df, n_splits=N_SPLITS):
    inner = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    a, b = next(inner.split(tr, groups=df.source.values[tr]))   # validácia podľa zdroja

    model, hist, best_epoch = train_finetune(tr[a], tr[b], seed=seed)
    pred = predict_logits(model, te).argmax(1).numpy()

    split_rows = split_metrics(seed, y[te], pred, ds[te])
    for r in split_rows:
        rows.append({**r, "best_epoch": best_epoch})
    y_all.append(y[te])
    p_all.append(pred)
    histories.append((hist, best_epoch))
    print(f"delenie {seed}: " + ", ".join(f"{r['test_set']} {r['bal_acc']:.3f}" for r in split_rows)
          + f", najlepšia epocha {best_epoch}")

    del model
    if device == "cuda":
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
y_all, p_all = np.concatenate(y_all), np.concatenate(p_all)

Path("results").mkdir(exist_ok=True)
results.to_csv("results/video_finetune.csv", index=False)
print(summarize(results, "R(2+1)D doladená").to_string())

In [ ]:
hist, best_epoch = histories[0]
plt.plot(range(1, len(hist) + 1), hist[:, 0], label="train")
plt.plot(range(1, len(hist) + 1), hist[:, 1], label="validácia")
plt.axvline(best_epoch, ls="--", c="gray", label="najlepšia epocha")
plt.xlabel("epocha")
plt.ylabel("strata (cross-entropy)")
plt.legend()
plt.show()

In [ ]:
print(classification_report(y_all, p_all, target_names=CLASSES))
ConfusionMatrixDisplay.from_predictions(y_all, p_all, display_labels=CLASSES, cmap="Blues");